# RAG-Only Baseline And Failure Modes

This notebook builds a local retriever and then uses a real provider call for grounded answer generation. RAG adds evidence to the prompt; it does not train model weights.


## Retrieval Failure Modes

A RAG system can fail before the model ever writes an answer. The retriever may miss the right document, return a weakly related document, or rank the correct source below irrelevant context. The generator can also fail after retrieval by ignoring evidence, dropping citations, or answering beyond the context.

For this reason, retrieval evaluation and generated-answer evaluation should stay separate. Recall@k tells us whether the expected document appears in the top results. Citation and abstention checks tell us whether the generation path respects the evidence that was retrieved.


## Retrieval Terms In Plain Language

A **query** is the text used to search the corpus. It can be the user question directly, or a rewritten version of it. A **chunk** is a smaller piece of a document that can be retrieved independently. A **score** estimates how similar or relevant a document is to the query. A **top-k result** is one of the first `k` records returned by the retriever.

The notebook uses simple token counts instead of embeddings so the mechanics stay visible. In production, the same concepts usually appear with embedding models and vector databases: query embedding, chunk embedding, similarity score, and top-k retrieval. The teaching point is the same: generation quality depends on whether the right evidence enters the prompt.


### Exercise

If the retriever misses the correct document, should the generator still answer confidently?


### Solution

No. It should abstain or ask for more context because the answer would not be grounded in available evidence.


## Visual Map: Retrieval Quality Comes Before Generation

RAG quality depends on whether the right evidence reaches the model. A fluent generated answer is still weak if retrieval missed the supporting passage.

**Semantic-search pipeline.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/semantic_search_pipeline.png). This visual shows the retrieval half of the system: encode the query, search indexed passages, then pass selected evidence downstream.

<img src="assets/hf-semantic-search-pipeline.png" alt="Semantic search pipeline" style="max-width: 760px; width: 100%; height: auto;">

---

After the pipeline view, the next visual focuses on tuning retrieval settings and measuring their impact.

**RAG settings and accuracy trade-off.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/RAG_settings_accuracy.png). This visual is useful because retrieval settings are measurable choices, not decoration around the LLM call.

<img src="assets/hf-cookbook-rag-settings-accuracy.png" alt="RAG settings and accuracy" style="max-width: 760px; width: 100%; height: auto;">

---

The last visual adds an operational concern: repeated or similar requests should not always require the full retrieval-and-generation path.

**Semantic cache architecture.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/semantic_cache.jpg). The cache example makes a practical point: once retrieval and generation are real API paths, latency and repeated-query behavior become part of the architecture discussion.

<img src="assets/hf-cookbook-semantic-cache.jpg" alt="Semantic cache architecture" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

In a RAG pipeline, should retrieval quality be checked before or after judging the generated answer?


### Solution

Check retrieval quality first. If the right evidence is missing, a fluent generated answer is not trustworthy.


In [ ]:
import json
import re
from pprint import pprint
from typing import Any, Literal

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ReplySource,
    call_model,
    format_context,
    retrieve,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Inspect The Transparent Local Retriever

[`strategy_helpers.py`](strategy_helpers.py) implements tokenization, term-frequency vectors, cosine ranking, and source-metadata preservation once for the repository. This notebook inspects that baseline before using it in the answer path.


In [ ]:
retrieval_preview = retrieve("duplicate payment refund invoice", top_k=3)
[(row["doc_id"], round(row["score"], 3)) for row in retrieval_preview]

### Exercise: Check The Indexed Corpus

Use the shared source list to count how many policy documents can be retrieved.


In [ ]:
# TODO: calculate the number of indexed policy documents.
indexed_document_count = None
indexed_document_count

### Solution


In [ ]:
indexed_document_count = len(DOCUMENTS)
indexed_document_count

## Retrieve Policy Evidence

The retriever ranks policy documents by cosine similarity between sparse token-count vectors.


In [ ]:
question = REQUESTS[1]["question"]
retrieved = retrieve(question, top_k=3)
pprint(retrieved)

In [ ]:
# Exercise: inspect the top retrieved document ID for this question.
# top_doc_id = ...


In [ ]:
# Solution
top_doc_id = retrieved[0]["doc_id"]
print(top_doc_id)

## Call The Model With Citations

The prompt requires the answer to cite source IDs. If retrieval is weak, the model should abstain rather than invent a policy.


The answer schema makes grounding inspectable: the output must say what it answered from and whether it abstained.


In [ ]:
class CitedAnswer(BaseModel):
    """Structured representation of a grounded answer."""

    answer: str
    source_ids: list[str]
    status: Literal["answered", "abstained", "parse_failed", "invalid-citations"]
    response_source: ReplySource | Literal["not_called"] = "not_called"

Retrieve first and abstain before calling the model when the best evidence is too weak.


In [ ]:
def retrieve_or_abstain(
    question: str, *, threshold: float = 0.08
) -> list[dict[str, Any]]:
    """Return evidence only when the top retrieval score is high enough."""
    evidence = retrieve(question, top_k=3)

    if not evidence or evidence[0]["score"] < threshold:
        return []

    return evidence

The RAG answer path validates both the JSON response and its citation IDs against the retrieved context. Valid IDs do not prove that the cited passages support every claim.


Build the cited-answer prompt separately from the RAG control flow.


In [ ]:
def rag_messages(question: str, evidence: list[dict[str, Any]]) -> list[dict[str, str]]:
    """Create a context-grounded prompt for cited JSON output."""
    context = format_context(evidence)
    return [
        {
            "role": "system",
            "content": (
                "Answer only from the context. Return JSON with answer, source_ids, and status. "
                "Cite context IDs inline as [doc_id] and list the same IDs in source_ids. "
                "Use status answered, or abstained with no citations if evidence is insufficient."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

The dry response mirrors the expected cited JSON structure.


In [ ]:
def dry_cited_answer(evidence: list[dict[str, Any]]) -> str:
    """Create deterministic cited output for credential-free execution."""
    return json.dumps(
        {
            "answer": f"{evidence[0]['text']} [{evidence[0]['doc_id']}]",
            "source_ids": [evidence[0]["doc_id"]],
            "status": "answered",
        }
    )

The final answer helper now only handles abstention, provider routing, and validation.


In [ ]:
def rag_answer(question: str, *, threshold: float = 0.08) -> CitedAnswer:
    """Retrieve policy context and ask the provider for a cited answer."""
    evidence = retrieve_or_abstain(question, threshold=threshold)
    if not evidence:
        return CitedAnswer(
            answer="I do not know from the available policy documents.",
            source_ids=[],
            status="abstained",
        )

    reply = call_model(
        rag_messages(question, evidence),
        dry_response=dry_cited_answer(evidence),
        response_format={"type": "json_object"},
    )

    try:
        result = CitedAnswer.model_validate_json(reply.text)
        result.response_source = reply.source
    except ValidationError:
        return CitedAnswer(
            answer=reply.text,
            source_ids=[],
            status="parse_failed",
            response_source=reply.source,
        )

    cited_ids = set(re.findall(r"\[([^\[\]]*)\]", result.answer))
    declared_ids = set(result.source_ids)
    context_ids = {record["doc_id"] for record in evidence}
    valid_citations = (
        bool(cited_ids) and cited_ids == declared_ids and cited_ids <= context_ids
        if result.status == "answered"
        else result.status == "abstained" and not (cited_ids or declared_ids)
    )
    if not valid_citations:
        return CitedAnswer(
            answer="The generated answer failed citation validation. Review the evidence.",
            source_ids=[],
            status="invalid-citations",
            response_source=reply.source,
        )
    return result

Run the grounded path once so the cited response can be inspected before evaluating retrieval quality.


In [ ]:
answer = rag_answer(question)
pprint(answer.model_dump())

## Evaluate Retrieval Recall

Before judging generation, check whether retrieval found the expected source. If the right document is absent, the answer is already constrained by poor evidence.


## What Recall@k Does And Does Not Tell You

**Recall@k** asks a narrow question: did the expected source appear somewhere in the top `k` retrieved records? If `recall@1` is low but `recall@3` is high, the retriever often finds the right document but ranks it below other documents. That may still be usable if the generator receives all three records, but it increases the risk of distraction.

Recall does not judge whether the final answer is well written, complete, or safe. It also does not prove that the cited sentence supports every claim in the answer. That is why RAG evaluation usually has at least two layers: retrieval metrics first, then answer-grounding checks.


In [ ]:
def recall_at_k(k: int = 3) -> float:
    """Measure whether expected policy documents appear in top-k retrieval."""
    hits = 0
    for request in REQUESTS:
        retrieved_ids = [
            record["doc_id"] for record in retrieve(request["question"], top_k=k)
        ]
        hits += request["expected_doc"] in retrieved_ids
    return hits / len(REQUESTS)


for k in [1, 2, 3]:
    print(f"recall@{k} = {recall_at_k(k):.2f}")

### Exercise

If `recall@2` is higher than `recall@1`, what changed?


### Solution

The correct document was not always ranked first, but it appeared when the retriever was allowed to return two documents.


In [ ]:
unsupported_question = "Where should I park my bicycle near campus?"
pprint(rag_answer(unsupported_question, threshold=0.18).model_dump())

## Further Reading

- [Sentence Transformers semantic search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html) for moving beyond the transparent sparse baseline.
- [Ragas retrieval metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/) for broader retrieval evaluation.
